# NHIS Data Download: Self-Reported Liver Disease & Linked Mortality

This notebook downloads NHIS Sample Adult files (2015–2018) and linked mortality
files, then harmonizes them into a single analysis-ready dataset.

**Why 2015–2018?**
- These years provide CSV-format public-use files with the `LIVEV` (ever had liver
  condition) variable, consistent covariate definitions, and linked mortality
  follow-up through December 31, 2019.
- The 2019 NHIS redesign changed variable names and rotating content schedules.
  The public-use linked mortality files currently cover survey years through 2018 only.

**Variables extracted:**

| Variable | Description | Source |
|----------|-------------|--------|
| LIVEV | Ever had liver condition | Sample Adult questionnaire |
| AGE_P | Age at interview | Demographics |
| SEX | Sex (1=Male, 2=Female) | Demographics |
| HISPAN_I | Hispanic origin | Demographics |
| RACERPI2 | Race | Demographics |
| BMI | Body mass index (coded ×100) | Sample Adult |
| DIBEV / DIBEV1 | Ever told had diabetes | Sample Adult |
| SMKEV | Ever smoked 100 cigarettes | Sample Adult |
| HYPEV | Ever told had hypertension | Sample Adult |
| MORTSTAT | Mortality status | Linked mortality file |
| UCOD_LEADING | Underlying cause of death | Linked mortality file |
| PERMTH_INT | Person-months from interview | Linked mortality file |

In [1]:
import os, io, zipfile, warnings
import numpy as np
import pandas as pd
import requests

warnings.filterwarnings('ignore', category=FutureWarning)

RAW_DIR = os.path.join(os.path.abspath('.'), 'data', 'nhis', 'raw')
DERIVED_DIR = os.path.join(os.path.abspath('.'), 'data', 'nhis', 'derived')
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(DERIVED_DIR, exist_ok=True)
print(f'Raw:     {RAW_DIR}')
print(f'Derived: {DERIVED_DIR}')

Raw:     /home/user/ai_assisted_research/nhanes_mortality_fibrosis/data/nhis/raw
Derived: /home/user/ai_assisted_research/nhanes_mortality_fibrosis/data/nhis/derived


## Configuration

In [2]:
YEARS = [2015, 2016, 2017, 2018]

# CDC FTP base URLs
# 2016-2018 have separate samadultcsv.zip; 2015 has csv inside samadult.zip
ADULT_URLS = {
    2015: 'https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Datasets/NHIS/2015/samadult.zip',
    2016: 'https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Datasets/NHIS/2016/samadultcsv.zip',
    2017: 'https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Datasets/NHIS/2017/samadultcsv.zip',
    2018: 'https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Datasets/NHIS/2018/samadultcsv.zip',
}

MORT_URL_TEMPLATE = (
    'https://ftp.cdc.gov/pub/health_statistics/NCHS/datalinkage/'
    'linked_mortality/NHIS_{year}_MORT_2019_PUBLIC.dat'
)

# Mortality file fixed-width spec (from CDC R read-in program)
MORT_COLSPEC = [
    (0, 14),   # PUBLICID (char)
    (14, 15),  # ELIGSTAT
    (15, 16),  # MORTSTAT
    (16, 19),  # UCOD_LEADING
    (19, 20),  # DIABETES (MCOD flag)
    (20, 21),  # HYPERTEN (MCOD flag)
    (21, 22),  # DODQTR
    (22, 26),  # DODYEAR
    (26, 34),  # WGT_NEW
    (34, 42),  # SA_WGT_NEW
]
MORT_NAMES = ['PUBLICID', 'ELIGSTAT', 'MORTSTAT', 'UCOD_LEADING',
              'DIABETES_MCOD', 'HYPERTEN_MCOD', 'DODQTR', 'DODYEAR',
              'WGT_NEW', 'SA_WGT_NEW']

## Download helper

In [3]:
def download_file(url, dest_path, retries=4):
    """Download a file with exponential backoff."""
    import time
    if os.path.exists(dest_path) and os.path.getsize(dest_path) > 1000:
        print(f'  Already exists: {os.path.basename(dest_path)}')
        return dest_path
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=120)
            r.raise_for_status()
            with open(dest_path, 'wb') as f:
                f.write(r.content)
            print(f'  Downloaded: {os.path.basename(dest_path)} ({len(r.content):,} bytes)')
            return dest_path
        except Exception as e:
            wait = 2 ** (attempt + 1)
            print(f'  Attempt {attempt+1} failed ({e}), retrying in {wait}s...')
            time.sleep(wait)
    raise RuntimeError(f'Failed to download {url} after {retries} attempts')

## Download and extract Sample Adult files

In [4]:
adult_csvs = {}

for year in YEARS:
    print(f'\n=== {year} ===')
    url = ADULT_URLS[year]
    zip_path = os.path.join(RAW_DIR, f'samadult{year}.zip')
    download_file(url, zip_path)
    
    # Extract CSV from zip
    csv_path = os.path.join(RAW_DIR, f'samadult{year}.csv')
    if not os.path.exists(csv_path):
        with zipfile.ZipFile(zip_path) as zf:
            # Find the .csv file inside
            csv_names = [n for n in zf.namelist() if n.lower().endswith('.csv')]
            if not csv_names:
                raise FileNotFoundError(f'No CSV in {zip_path}: {zf.namelist()}')
            # Extract and rename
            with zf.open(csv_names[0]) as src, open(csv_path, 'wb') as dst:
                dst.write(src.read())
        print(f'  Extracted: {csv_names[0]} -> {os.path.basename(csv_path)}')
    else:
        print(f'  Already extracted: {os.path.basename(csv_path)}')
    
    adult_csvs[year] = csv_path

print(f'\nDownloaded {len(adult_csvs)} Sample Adult files.')


=== 2015 ===
  Already exists: samadult2015.zip
  Already extracted: samadult2015.csv

=== 2016 ===
  Already exists: samadult2016.zip
  Already extracted: samadult2016.csv

=== 2017 ===
  Already exists: samadult2017.zip
  Already extracted: samadult2017.csv

=== 2018 ===
  Already exists: samadult2018.zip
  Already extracted: samadult2018.csv

Downloaded 4 Sample Adult files.


## Download linked mortality files

In [5]:
mort_files = {}

for year in YEARS:
    print(f'\n=== {year} ===')
    url = MORT_URL_TEMPLATE.format(year=year)
    dat_path = os.path.join(RAW_DIR, f'NHIS_{year}_MORT_2019_PUBLIC.dat')
    download_file(url, dat_path)
    mort_files[year] = dat_path

print(f'\nDownloaded {len(mort_files)} linked mortality files.')


=== 2015 ===
  Already exists: NHIS_2015_MORT_2019_PUBLIC.dat

=== 2016 ===
  Already exists: NHIS_2016_MORT_2019_PUBLIC.dat

=== 2017 ===
  Already exists: NHIS_2017_MORT_2019_PUBLIC.dat

=== 2018 ===
  Already exists: NHIS_2018_MORT_2019_PUBLIC.dat

Downloaded 4 linked mortality files.


## Parse and harmonize each year

In [6]:
def parse_mortality(dat_path):
    """Parse the fixed-width linked mortality file."""
    df = pd.read_fwf(dat_path, colspecs=MORT_COLSPEC, names=MORT_NAMES,
                     dtype={'PUBLICID': str}, na_values=['.', ''])
    # Convert numeric columns
    for col in ['ELIGSTAT', 'MORTSTAT', 'UCOD_LEADING', 'DIABETES_MCOD',
                'HYPERTEN_MCOD', 'DODQTR', 'DODYEAR']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def build_publicid(row, year):
    """Construct PUBLICID = SRVY_YR(4) + HHX(6, zero-padded) + FMX(2) + FPX(2)."""
    return f"{year:04d}{int(row['HHX']):06d}{int(row['FMX']):02d}{int(row['FPX']):02d}"


def harmonize_year(year):
    """Load, harmonize, and merge one NHIS survey year."""
    print(f'\n=== {year} ===')
    
    # --- Load Sample Adult ---
    csv_path = adult_csvs[year]
    df = pd.read_csv(csv_path, low_memory=False)
    print(f'  Raw rows: {len(df):,}')
    
    # --- Harmonize variable names across years ---
    # Diabetes: DIBEV in 2015, DIBEV1 in 2016-2018
    if 'DIBEV1' in df.columns and 'DIBEV' not in df.columns:
        df.rename(columns={'DIBEV1': 'DIBEV'}, inplace=True)
    
    # Survey design: STRAT_P/PSU_P in 2015, PSTRAT/PPSU in 2016-2018
    if 'STRAT_P' in df.columns and 'PSTRAT' not in df.columns:
        df.rename(columns={'STRAT_P': 'PSTRAT', 'PSU_P': 'PPSU'}, inplace=True)
    
    # --- Select and recode covariates ---
    out = pd.DataFrame()
    out['HHX'] = df['HHX']
    out['FMX'] = df['FMX']
    out['FPX'] = df['FPX']
    out['YEAR'] = year
    
    # Demographics
    out['AGE'] = df['AGE_P'].copy()
    out.loc[out['AGE'] >= 85, 'AGE'] = 85  # top-code at 85 (consistent with mortality file)
    out['FEMALE'] = (df['SEX'] == 2).astype(int)
    
    # Race/ethnicity -> single variable (similar to NHANES RIDRETH1)
    # HISPAN_I: 0-11 = Hispanic subcategories, 12 = Not Hispanic
    # RACERPI2: 1=White, 2=Black, 3=AIAN, 4=Asian, 5=Race group not releasable, 6=Multiple
    out['HISPANIC'] = (df['HISPAN_I'] < 12).astype(int)
    out['RACE_ETH'] = 'Other'
    out.loc[(df['HISPAN_I'] < 12), 'RACE_ETH'] = 'Hispanic'
    out.loc[(df['HISPAN_I'] == 12) & (df['RACERPI2'] == 1), 'RACE_ETH'] = 'NH White'
    out.loc[(df['HISPAN_I'] == 12) & (df['RACERPI2'] == 2), 'RACE_ETH'] = 'NH Black'
    out.loc[(df['HISPAN_I'] == 12) & (df['RACERPI2'] == 4), 'RACE_ETH'] = 'NH Asian'
    
    # BMI (coded as BMI × 100, 9999 = unknown)
    bmi_raw = df['BMI'].copy()
    bmi_raw[bmi_raw >= 9990] = np.nan
    out['BMI'] = bmi_raw / 100.0
    
    # Self-reported conditions (1=Yes, 2=No; 7/8/9 = Refused/Not ascertained/DK)
    out['LIVER_EVER'] = np.where(df['LIVEV'] == 1, 1,
                        np.where(df['LIVEV'] == 2, 0, np.nan))
    out['DIABETES'] = np.where(df['DIBEV'].isin([1, 3]), 1,  # 1=Yes, 3=Borderline
                      np.where(df['DIBEV'] == 2, 0, np.nan))
    out['SMOKE_EVER'] = np.where(df['SMKEV'] == 1, 1,
                        np.where(df['SMKEV'] == 2, 0, np.nan))
    out['HYPERTENSION'] = np.where(df['HYPEV'] == 1, 1,
                          np.where(df['HYPEV'] == 2, 0, np.nan))
    
    # Survey design
    out['WTFA_SA'] = df['WTFA_SA']
    out['PSTRAT'] = df['PSTRAT']
    out['PPSU'] = df['PPSU']
    
    # --- Construct PUBLICID and merge mortality ---
    out['PUBLICID'] = [build_publicid(row, year) for _, row in
                       df[['HHX', 'FMX', 'FPX']].iterrows()]
    
    mort = parse_mortality(mort_files[year])
    print(f'  Mortality records: {len(mort):,}')
    
    merged = out.merge(mort[['PUBLICID', 'ELIGSTAT', 'MORTSTAT', 'UCOD_LEADING',
                             'DODQTR', 'DODYEAR', 'SA_WGT_NEW']],
                       on='PUBLICID', how='left')
    
    # Compute person-months from interview to death or end of follow-up (Dec 31, 2019).
    # NHIS interviews run Jan-Dec; approximate interview at mid-year (July).
    # For decedents: use DODYEAR + DODQTR for quarter-level precision.
    # DODQTR: 1=Jan-Mar, 2=Apr-Jun, 3=Jul-Sep, 4=Oct-Dec -> midpoint months 2,5,8,11
    qtr_month = merged['DODQTR'].map({1: 2, 2: 5, 3: 8, 4: 11})
    merged['PERMTH_INT'] = np.where(
        merged['MORTSTAT'] == 1,
        (merged['DODYEAR'] - year) * 12 + (qtr_month.fillna(6) - 7),
        (2019 - year) * 12 + 6  # alive through Dec 2019, ~6 months from mid-year interview
    )
    merged['PERMTH_INT'] = merged['PERMTH_INT'].clip(lower=0)
    
    # Filter to eligible adults (ELIGSTAT == 1)
    eligible = merged[merged['ELIGSTAT'] == 1].copy()
    print(f'  Eligible for mortality linkage: {len(eligible):,}')
    print(f'  Deaths: {(eligible["MORTSTAT"]==1).sum():,}')
    print(f'  Liver condition (LIVEV=1): {(eligible["LIVER_EVER"]==1).sum():,}')
    
    return eligible

In [7]:
frames = []
for year in YEARS:
    df = harmonize_year(year)
    frames.append(df)

pooled = pd.concat(frames, ignore_index=True)
print(f'\n=== Pooled ===')
print(f'Total eligible adults: {len(pooled):,}')
print(f'Total deaths: {(pooled["MORTSTAT"]==1).sum():,}')
print(f'Liver condition: {(pooled["LIVER_EVER"]==1).sum():,}')
print(f'Years: {sorted(pooled["YEAR"].unique())}')
print(f'PERMTH_INT range: {pooled["PERMTH_INT"].min():.0f}\u2013{pooled["PERMTH_INT"].max():.0f}')


=== 2015 ===


  Raw rows: 33,672


  Mortality records: 45,963
  Eligible for mortality linkage: 33,129
  Deaths: 1,851
  Liver condition (LIVEV=1): 428

=== 2016 ===


  Raw rows: 33,028


  Mortality records: 44,135
  Eligible for mortality linkage: 32,487
  Deaths: 1,456
  Liver condition (LIVEV=1): 496

=== 2017 ===


  Raw rows: 26,742


  Mortality records: 35,587
  Eligible for mortality linkage: 26,267
  Deaths: 832
  Liver condition (LIVEV=1): 380

=== 2018 ===


  Raw rows: 25,417


  Mortality records: 33,686
  Eligible for mortality linkage: 24,922
  Deaths: 451
  Liver condition (LIVEV=1): 417

=== Pooled ===
Total eligible adults: 116,805
Total deaths: 4,590
Liver condition: 1,721
Years: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018)]
PERMTH_INT range: 0–54


## Save derived dataset

In [8]:
out_path = os.path.join(DERIVED_DIR, 'nhis_2015_2018_pooled.parquet')
pooled.to_parquet(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Shape: {pooled.shape}')
print(f'Columns: {list(pooled.columns)}')

Saved: /home/user/ai_assisted_research/nhanes_mortality_fibrosis/data/nhis/derived/nhis_2015_2018_pooled.parquet
Shape: (116805, 24)
Columns: ['HHX', 'FMX', 'FPX', 'YEAR', 'AGE', 'FEMALE', 'HISPANIC', 'RACE_ETH', 'BMI', 'LIVER_EVER', 'DIABETES', 'SMOKE_EVER', 'HYPERTENSION', 'WTFA_SA', 'PSTRAT', 'PPSU', 'PUBLICID', 'ELIGSTAT', 'MORTSTAT', 'UCOD_LEADING', 'DODQTR', 'DODYEAR', 'SA_WGT_NEW', 'PERMTH_INT']


## Quick checks

In [9]:
print('Variable summaries:\n')
for col in ['AGE', 'FEMALE', 'BMI', 'LIVER_EVER', 'DIABETES',
            'SMOKE_EVER', 'HYPERTENSION', 'MORTSTAT', 'UCOD_LEADING', 'PERMTH_INT']:
    if col in pooled.columns:
        valid = pooled[col].notna().sum()
        if pooled[col].nunique() < 10:
            print(f'{col} (n={valid:,}):')
            print(f'  {pooled[col].value_counts().sort_index().to_dict()}')
        else:
            print(f'{col} (n={valid:,}): mean={pooled[col].mean():.1f}, '
                  f'median={pooled[col].median():.1f}, '
                  f'range={pooled[col].min():.0f}\u2013{pooled[col].max():.0f}')
        print()

Variable summaries:

AGE (n=116,805): mean=50.8, median=51.0, range=18–85

FEMALE (n=116,805):
  {0: 52821, 1: 63984}



BMI (n=113,028): mean=28.1, median=27.0, range=12–97

LIVER_EVER (n=114,878):
  {0.0: 113157, 1.0: 1721}

DIABETES (n=116,713):
  {0.0: 101024, 1.0: 15689}

SMOKE_EVER (n=116,438):
  {0.0: 69730, 1.0: 46708}

HYPERTENSION (n=116,647):
  {0.0: 75205, 1.0: 41442}

MORTSTAT (n=116,805):
  {0.0: 112215, 1.0: 4590}

UCOD_LEADING (n=4,590):
  {1.0: 1144, 2.0: 1065, 3.0: 285, 4.0: 220, 5.0: 247, 10.0: 1629}

PERMTH_INT (n=116,805): mean=36.8, median=42.0, range=0–54



In [10]:
# Cross-tabulate liver condition and mortality
ct = pd.crosstab(pooled['LIVER_EVER'].map({1: 'Liver+', 0: 'Liver\u2212'}),
                 pooled['MORTSTAT'].map({1: 'Dead', 0: 'Alive'}),
                 margins=True)
print('Liver condition \u00d7 Mortality status:')
ct

Liver condition × Mortality status:


MORTSTAT,Alive,Dead,All
LIVER_EVER,,,
Liver+,1508,213,1721
Liver−,108877,4280,113157
All,110385,4493,114878
